In [1]:
import os 
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] ="1"

from datasets import load_dataset
import json
import re

OUTPUT_PATH = r"C:\datasets\llm-data\sft"
os.makedirs(OUTPUT_PATH, exist_ok=True)

In [2]:
DS_KEYWORDS = [
    "python", "pandas", "numpy", "matplotlib", "sklearn", "scikit",
    "regression", "classification", "clustering", "neural", "deep learning",
    "machine learning", "data", "model", "training", "overfitting",
    "gradient", "loss", "accuracy", "dataset", "feature", "plot",
    "visualization", "seaborn", "tensorflow", "pytorch", "statistics"
]

In [3]:
def is_ds_related(text):
    text = text.lower()
    return any(kw in text for kw in DS_KEYWORDS)

In [4]:
def to_alpaca(instruction,output,input_text=""):
    return {
        "instruction" : instruction.strip(),
        "output"      : output.strip(),
        "input"       : input_text.strip()    
        }

In [5]:
ds1 = load_dataset("iamtarun/python_code_instructions_18k_alpaca", split="train")
filtered1 = []
for item in ds1:
    if is_ds_related(item.get("instruction", "") + item.get("output", "")):
        filtered1.append(to_alpaca(item["instruction"], item["output"], item.get("input", "")))

README.md:   0%|          | 0.00/905 [00:00<?, ?B/s]

c:\Users\Ayush\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Ayush\.cache\huggingface\hub\datasets--iamtarun--python_code_instructions_18k_alpaca. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


data/train-00000-of-00001-8b6e212f3e1ece(…):   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/18612 [00:00<?, ? examples/s]

In [6]:
ds2 = load_dataset("sahil2801/CodeAlpaca-20k", split="train")
filtered2 = []
for item in ds2:
    if is_ds_related(item.get("instruction", "") + item.get("output", "")):
        filtered2.append(to_alpaca(item["instruction"], item["output"], item.get("input", "")))


README.md:   0%|          | 0.00/147 [00:00<?, ?B/s]

c:\Users\Ayush\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Ayush\.cache\huggingface\hub\datasets--sahil2801--CodeAlpaca-20k. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


code_alpaca_20k.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/20022 [00:00<?, ? examples/s]

In [7]:
print(f"✓ Dataset 1: {len(filtered1):,} DS-related samples")
print(f"✓ Dataset 2: {len(filtered2):,} DS-related samples")

✓ Dataset 1: 18,612 DS-related samples
✓ Dataset 2: 4,541 DS-related samples


In [8]:
combined = filtered1 + filtered2
print(f"✓ Total HuggingFace samples: {len(combined):,}")


✓ Total HuggingFace samples: 23,153


In [9]:
with open(f"{OUTPUT_PATH}/hf_sft.json", "w") as f:
    json.dump(combined, f, indent=2)